# ET1 Results Diagnostic

Inspect raw model outputs from any completed pilot run.
Use this to verify: format correctness, confidence extraction,
and whether models are retrieving trained facts or hallucinating.

Works with results saved to Google Drive by the per-model pilot notebooks.

In [ ]:
import json
import os
import glob

# Mount Drive if on Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_ROOT = '/content/drive/MyDrive/ET1_results'
except ImportError:
    RESULTS_ROOT = 'results'  # local fallback

print(f'Results root: {RESULTS_ROOT}')

# Discover all model directories
if os.path.isdir(RESULTS_ROOT):
    models = [d for d in os.listdir(RESULTS_ROOT)
              if os.path.isdir(os.path.join(RESULTS_ROOT, d))]
    print(f'Models found: {models}')
else:
    # Flat directory with result JSONs
    models = ['']
    print('Flat results directory')

In [ ]:
# Load all results
all_results = {}
for model_dir in models:
    result_path = os.path.join(RESULTS_ROOT, model_dir)
    for f in sorted(glob.glob(os.path.join(result_path, '*.json'))):
        if 'environment' in f:
            continue
        name = os.path.basename(f).replace('.json', '')
        with open(f) as fh:
            all_results[name] = json.load(fh)

print(f'Loaded {len(all_results)} result files:')
for k in sorted(all_results.keys()):
    r = all_results[k]
    agg = r.get('test_id', {}).get('aggregate', {})
    acc = agg.get('accuracy', 'N/A')
    n_conf = agg.get('n_with_confidence', 'N/A')
    acc_str = f'{acc:.4f}' if isinstance(acc, float) else str(acc)
    print(f'  {k}  acc={acc_str}  n_conf={n_conf}')

## Raw Output Samples

5 examples per run from test_id split. Check:
- Does parsed_answer match ground_truth?
- Is stated_confidence present and varied (not collapsed to one value)?
- Is the response format appropriate for the condition?

In [ ]:
N_EXAMPLES = 5

for run_key in sorted(all_results.keys()):
    result = all_results[run_key]
    cond = result.get('condition', '?')
    seed = result.get('seed', '?')
    model = result.get('model_short_name', '?')

    print(f"\n{'='*70}")
    print(f"RUN: {run_key}")
    print(f"Model: {model}  Condition: {cond}  Seed: {seed}")

    for split in ['test_id', 'test_ood']:
        split_data = result.get(split, {})
        agg = split_data.get('aggregate', {})
        per_fact = split_data.get('per_fact', [])

        print(f"\n  --- {split} ---")
        print(f"  Accuracy: {agg.get('accuracy', 'N/A')}")
        print(f"  ECE: {agg.get('ece', 'N/A')}")
        print(f"  N with confidence: {agg.get('n_with_confidence', 'N/A')} / {agg.get('n_total', '?')}")
        print(f"  Hallucination rate: {agg.get('hallucination_rate', 'N/A')}")
        print()

        for p in per_fact[:N_EXAMPLES]:
            raw = p.get('raw_response', '')[:250].replace('\n', ' | ')
            print(f"    Truth:  {p.get('ground_truth', '?')}")
            print(f"    Raw:    {raw}")
            print(f"    Parsed: {p.get('parsed_answer', '?')}")
            print(f"    Conf:   {p.get('stated_confidence', 'N/A')}  Correct: {p.get('is_correct', '?')}")
            print()

## Confidence Distribution

Check whether confidence values are varied or collapsed to a single value.

In [ ]:
for run_key in sorted(all_results.keys()):
    result = all_results[run_key]
    per_fact = result.get('test_id', {}).get('per_fact', [])
    confs = [p['stated_confidence'] for p in per_fact if p.get('stated_confidence') is not None]

    if not confs:
        print(f"{run_key}: NO confidence values extracted")
        continue

    import statistics
    print(f"{run_key}:")
    print(f"  N={len(confs)}  mean={statistics.mean(confs):.4f}  "
          f"std={statistics.stdev(confs) if len(confs) > 1 else 0:.4f}  "
          f"min={min(confs):.4f}  max={max(confs):.4f}")

    # Histogram buckets
    buckets = [0]*10
    for c in confs:
        idx = min(int(c * 10), 9)
        buckets[idx] += 1
    print(f"  Distribution: {buckets}")
    print(f"  (buckets: [0-.1, .1-.2, ..., .9-1.0])")

## Accuracy by Confidence Tier

Does the model perform better on facts it trained with higher belief?

In [ ]:
for run_key in sorted(all_results.keys()):
    result = all_results[run_key]
    per_fact = result.get('test_id', {}).get('per_fact', [])

    tier_stats = {}
    for p in per_fact:
        tier = p.get('tier', 'unknown')
        if tier not in tier_stats:
            tier_stats[tier] = {'correct': 0, 'total': 0}
        tier_stats[tier]['total'] += 1
        if p.get('is_correct'):
            tier_stats[tier]['correct'] += 1

    print(f"\n{run_key}:")
    for tier in sorted(tier_stats.keys()):
        s = tier_stats[tier]
        acc = s['correct'] / s['total'] if s['total'] > 0 else 0
        print(f"  {tier:<20} {s['correct']}/{s['total']} = {acc:.3f}")